# 03 · Calibración SOL

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/telmomm/nanovna-measurement-course/HEAD?labpath=es%2Fnotebooks%2F03_calibracion_sol.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/telmomm/nanovna-measurement-course/blob/main/es/notebooks/03_calibracion_sol.ipynb)

En el cuaderno 02 viste que un instrumento distorsiona cada reflexión a través de tres términos de error. La calibración es la forma de hallar esos tres términos y deshacerlos. En el instrumento es un menú: conecta el cortocircuito, pulsa un botón; conecta el abierto, pulsa un botón; conecta la carga, pulsa un botón. Aquí haces lo que ocurre detrás de esos botones.

## Objetivos de aprendizaje

Al terminar este cuaderno serás capaz de:

- convertir el modelo de error de tres términos en un sistema de tres ecuaciones lineales y resolverlo;
- corregir una medida en bruto con los términos de error que has hallado;
- comprobar una calibración con datos que no se usaron para calcularla;
- explicar lo que una calibración *no* corrige: el plano de referencia y los estándares imperfectos.

**Necesitas:** el modelo de error de tres términos del [cuaderno 02](02_errores_sistematicos.ipynb). Sin hardware: las medidas reales están incluidas.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/telmomm/nanovna-measurement-course

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import skrf as rf

from vnacourse import plotting
from vnacourse.data import load_repetitions
from vnacourse.errormodel import ErrorTerms, apply_error_model, synthetic_error_terms
from vnacourse.reflection import gamma_to_z, z_to_gamma

plotting.use_course_style()

## 1. Tres incógnitas, tres ecuaciones

El modelo de error, a una frecuencia, es

$$\Gamma_m = e_{00} + \frac{e_{10}e_{01}\,\Gamma}{1 - e_{11}\,\Gamma}$$

Tiene tres incógnitas complejas. Si mides un dispositivo cuya $\Gamma$ real ya conoces, obtienes una ecuación. Tres dispositivos conocidos, llamados **estándares de calibración**, dan tres ecuaciones. La elección habitual es un cortocircuito (**S**hort, $\Gamma = -1$), un abierto (**O**pen, $\Gamma = +1$) y una carga (**L**oad, $\Gamma = 0$): tres puntos tan separados como es posible en el plano $\Gamma$.

Tal como está escrita, la ecuación no es lineal en los términos de error, porque $e_{11}$ está en un denominador. Multiplica ambos lados por $(1 - e_{11}\Gamma)$ y agrupa:

$$\Gamma_m = e_{00} + \Gamma\,\Gamma_m\,e_{11} - \Gamma\,\Delta_e
\qquad\text{con}\qquad \Delta_e = e_{00}e_{11} - e_{10}e_{01}$$

Ahora *sí* es lineal en las tres incógnitas $e_{00}$, $e_{11}$ y $\Delta_e$. Escribiéndola una vez por estándar:

$$
\begin{bmatrix}
1 & \Gamma_1\Gamma_{m1} & -\Gamma_1 \\
1 & \Gamma_2\Gamma_{m2} & -\Gamma_2 \\
1 & \Gamma_3\Gamma_{m3} & -\Gamma_3
\end{bmatrix}
\begin{bmatrix} e_{00} \\ e_{11} \\ \Delta_e \end{bmatrix}
=
\begin{bmatrix} \Gamma_{m1} \\ \Gamma_{m2} \\ \Gamma_{m3} \end{bmatrix}
$$

Resuelve este sistema 3×3 a cada frecuencia y recupera después el seguimiento como $e_{10}e_{01} = e_{00}e_{11} - \Delta_e$.

**Corrección.** Una vez conocidos los términos, despeja $\Gamma$ del modelo:

$$\Gamma = \frac{\Gamma_m - e_{00}}{e_{10}e_{01} + e_{11}\,(\Gamma_m - e_{00})}$$

Léelo de dentro hacia fuera: resta la directividad y después deshaz a la vez el seguimiento y la desadaptación de fuente.

## 2. Escribe el resolvedor

`measured` y `actual` contienen cada uno tres vectores (uno por estándar), con un valor por frecuencia. El bucle construye y resuelve el sistema 3×3 frecuencia a frecuencia.

In [ ]:
def solve_error_terms(measured, actual):
    """Términos de error a partir de tres estándares: lo que se midió y lo que son en realidad."""
    n_freq = len(measured[0])
    e00 = np.empty(n_freq, dtype=complex)
    e11 = np.empty(n_freq, dtype=complex)
    delta = np.empty(n_freq, dtype=complex)
    for k in range(n_freq):
        m = np.array([standard[k] for standard in measured])
        g = np.array([standard[k] for standard in actual])
        matrix = np.column_stack([np.ones(3), g * m, -g])
        e00[k], e11[k], delta[k] = np.linalg.solve(matrix, m)
    return ErrorTerms(directivity=e00, source_match=e11, tracking=e00 * e11 - delta)


def correct(gamma_measured, terms):
    """Elimina los errores sistemáticos de un coeficiente de reflexión medido."""
    e00, e11, tracking = terms
    return (gamma_measured - e00) / (tracking + e11 * (gamma_measured - e00))

## 3. Calibra el instrumento sintético

Empieza donde conoces la respuesta correcta. Toma el instrumento sintético del cuaderno 02 y «conecta» un cortocircuito, un abierto y una carga perfectos.

In [ ]:
f_hz = np.linspace(50e3, 1.45e9, 401)
f_mhz = f_hz / 1e6
true_terms = synthetic_error_terms(f_hz)

ones = np.ones(f_hz.shape)
ideal_sol = [-ones, ones, 0 * ones]  # cortocircuito, abierto, carga

measured_sol = [apply_error_model(gamma, true_terms) for gamma in ideal_sol]
found_terms = solve_error_terms(measured_sol, ideal_sol)

for name, found, true in zip(ErrorTerms._fields, found_terms, true_terms, strict=True):
    print(f"{name:<13} mayor diferencia con el término real: {np.abs(found - true).max():.1e}")

Los tres términos se recuperan hasta el error de redondeo. El instrumento nunca nos dijo sus términos de error: los hemos deducido de cómo distorsionaba tres dispositivos que conocíamos.

Ahora mide la resistencia de 75 Ω con el mismo instrumento y corrígela.

> **Predice antes de ejecutar.** En el cuaderno 02 la traza medida de esta resistencia oscilaba aproximadamente entre 0.09 y 0.27. ¿Cómo será la traza corregida?

In [ ]:
gamma_true = np.full(f_hz.shape, z_to_gamma(75))
gamma_raw = apply_error_model(gamma_true, true_terms)
gamma_corrected = correct(gamma_raw, found_terms)

fig, ax = plt.subplots()
ax.plot(f_mhz, np.abs(gamma_raw), color=plotting.SERIES[1], label="en bruto")
ax.plot(f_mhz, np.abs(gamma_corrected), color=plotting.SERIES[0], label="corregida")
ax.set(xlabel="frecuencia (MHz)", ylabel="$|\\Gamma|$", ylim=(0, 0.35))
ax.set_title("Resistencia de 75 Ω, instrumento sintético")
ax.legend(loc="lower right")

assert np.allclose(gamma_corrected, gamma_true)
print(
    f"Resistencia corregida: de {gamma_to_z(gamma_corrected).real.min():.3f} Ω "
    f"a {gamma_to_z(gamma_corrected).real.max():.3f} Ω"
)

El rizado ha desaparecido y la resistencia vale 75 Ω a todas las frecuencias. Con datos sintéticos el resultado es exacto, porque el instrumento cumple el modelo a la perfección y los estándares son perfectos. Los datos reales no serán tan amables.

El bucle anterior está escrito para leerse. El paquete del curso tiene el mismo resolvedor sin bucle, `vnacourse.calibration.solve_error_terms`, que resuelve todas las frecuencias en una llamada. Dan la misma respuesta:

In [ ]:
from vnacourse import calibration

package_terms = calibration.solve_error_terms(measured_sol, ideal_sol)
assert all(np.allclose(a, b) for a, b in zip(found_terms, package_terms, strict=True))
assert np.allclose(calibration.correct(gamma_raw, package_terms), gamma_corrected)
print("El mismo resultado que vnacourse.calibration")

## 4. Calibra medidas reales

El repositorio incluye las lecturas en bruto de un cortocircuito, un abierto y una carga tomadas con un NanoVNA-F V2, diez repeticiones de cada uno (véase [`data/README.md`](../data/README.md)). Promediar las repeticiones reduce el error aleatorio antes de despejar el sistemático.

De momento suponemos que los tres estándares son ideales. La sección 6 vuelve sobre esta suposición.

In [ ]:
f_real, short_reps = load_repetitions("standards", "short")
_, open_reps = load_repetitions("standards", "open")
_, load_reps = load_repetitions("standards", "load")
f_real_mhz = f_real / 1e6

ideal = [-1.0, 1.0, 0.0]  # cortocircuito, abierto, carga
measured = [reps.mean(axis=0) for reps in (short_reps, open_reps, load_reps)]
terms = calibration.solve_error_terms(measured, ideal)

fig, ax = plt.subplots()
ax.semilogy(f_real_mhz, np.abs(terms.directivity), label="directividad $|e_{00}|$")
ax.semilogy(f_real_mhz, np.abs(terms.source_match), label="desadapt. de fuente $|e_{11}|$")
ax.semilogy(f_real_mhz, np.abs(terms.tracking), label="seguimiento $|e_{10}e_{01}|$")
ax.set(xlabel="frecuencia (MHz)", ylabel="módulo (escala logarítmica)")
ax.set_title("Términos de error del NanoVNA-F V2, a partir de los estándares medidos")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)

above = f_real > 150e6
print(
    f"Módulo del seguimiento: hasta {np.abs(terms.tracking).max():.2f} por debajo de 150 MHz, "
    f"de {np.abs(terms.tracking[above]).min():.3f} a {np.abs(terms.tracking[above]).max():.3f} "
    "por encima"
)
print(
    f"Mayor desadaptación de fuente: {np.abs(terms.source_match).max():.3f} "
    f"a {f_real_mhz[np.abs(terms.source_match).argmax()]:.0f} MHz"
)

Esta es la huella del propio instrumento. Por debajo de unos 150 MHz el término de seguimiento está lejos de 1, y eso es lo que hacía que el abierto y el cortocircuito leyeran por encima de 1 en el cuaderno 02. Por encima de 150 MHz el seguimiento se mantiene a unos pocos por ciento de 1 y los otros dos términos son pequeños: ahí la lectura en bruto ya está cerca, y la calibración es un ajuste fino.

### 4.1 Corrige dos resistencias

El mismo instrumento midió una resistencia de 33 Ω y otra de 75 Ω, tres repeticiones de cada una. Sus coeficientes de reflexión reales son $(33-50)/(33+50) = -0.20$ y $(75-50)/(75+50) = +0.20$.

In [ ]:
duts = {}
for label, folder in (("33 Ω", "load_33ohm"), ("75 Ω", "load_75ohm")):
    _, reps = load_repetitions("dut", folder)
    raw = reps.mean(axis=0)
    duts[label] = {"raw": raw, "corrected": calibration.correct(raw, terms)}

low = f_real < 10e6
for label, dut in duts.items():
    z_raw = gamma_to_z(dut["raw"][low]).real
    z_corrected = gamma_to_z(dut["corrected"][low]).real
    print(
        f"Resistencia de {label} por debajo de 10 MHz:  "
        f"en bruto de {z_raw.min():6.1f} a {z_raw.max():6.1f} Ω   "
        f"corregida de {z_corrected.min():5.1f} a {z_corrected.max():5.1f} Ω"
    )

A baja frecuencia las lecturas en bruto no sirven y las corregidas caen a un par de ohmios de los valores nominales. La calibración funciona.

Mira ahora toda la banda.

> **Predice antes de ejecutar.** Una resistencia tiene la misma resistencia a todas las frecuencias. ¿Cómo debería ser el $\operatorname{Re} Z$ corregido de 50 kHz a 1.45 GHz?

In [ ]:
fig, ax = plt.subplots()
for label, dut in duts.items():
    ax.plot(f_real_mhz, gamma_to_z(dut["corrected"]).real, label=f"resistencia de {label}")
ax.set(xlabel="frecuencia (MHz)", ylabel="resistencia, Re Z (Ω)", ylim=(20, 95))
ax.set_title("Resistencia corregida en toda la banda")
ax.legend(loc="upper right", ncol=2);

### 4.2 El plano de referencia

Las dos resistencias oscilan aproximadamente entre 33 Ω y 75 Ω, en sentidos opuestos. ¿Está mal la calibración? Mira los mismos datos corregidos en la carta de Smith, y su módulo.

In [ ]:
fig, (ax_smith, ax_mag) = plt.subplots(1, 2, figsize=(10, 4.6))
plotting.gamma_plane(ax_smith, title="Γ corregido en la carta de Smith")
for label, dut in duts.items():
    gamma = dut["corrected"]
    ax_smith.plot(gamma.real, gamma.imag, label=f"resistencia de {label}")
    ax_mag.plot(f_real_mhz, np.abs(gamma), label=f"resistencia de {label}")
ax_smith.legend(loc="upper left", bbox_to_anchor=(0.0, 0.0))
ax_mag.set(xlabel="frecuencia (MHz)", ylabel="$|\\Gamma|$", ylim=(0, 0.4))
ax_mag.set_title("|Γ| corregido")
ax_mag.legend(loc="lower left");

El módulo se mantiene en torno a 0.2, como debe, con un rizado que crece con la frecuencia. Lo que más cambia es la **fase**: cada traza da varias vueltas alrededor del centro de la carta. Un coeficiente de reflexión de módulo aproximadamente constante cuya fase cae de forma continua con la frecuencia es lo que ves cuando el dispositivo está al final de un tramo de línea de transmisión. Una onda que recorre una línea de retardo $\tau$ en un sentido, se refleja y vuelve, llega con una fase adicional $-2\pi f \cdot 2\tau$:

$$\Gamma_\text{vista} = \Gamma_\text{dispositivo}\; e^{-j 4\pi f \tau}$$

La calibración lleva la medida al **plano de referencia**: el punto exacto donde se conectaron los estándares. Todo lo que hay más allá de ese punto (un adaptador, un cable corto, las patillas del componente) es, para el instrumento, parte del dispositivo. Los datos son coherentes con una línea corta entre el plano de calibración y cada resistencia.

Podemos estimar su retardo a partir de la pendiente de la fase y después eliminarlo.

In [ ]:
c0 = 299_792_458.0

fig, ax = plt.subplots()
for label, dut in duts.items():
    gamma = dut["corrected"]
    slope = np.polyfit(f_real, np.unwrap(np.angle(gamma)), 1)[0]  # rad por Hz
    tau = -slope / (4 * np.pi)
    dut["deembedded"] = gamma * np.exp(1j * 4 * np.pi * f_real * tau)

    z = gamma_to_z(dut["deembedded"]).real
    ax.plot(f_real_mhz, z, label=f"resistencia de {label}")
    print(
        f"{label}: retardo en un sentido {tau * 1e12:.0f} ps, "
        f"unos {tau * c0 * 0.7 * 100:.0f} cm de línea "
        f"a 0.7 c;  Re Z tras eliminarlo: media {z.mean():.1f} Ω, "
        f"de {z.min():.1f} a {z.max():.1f} Ω"
    )
ax.set(xlabel="frecuencia (MHz)", ylabel="resistencia, Re Z (Ω)", ylim=(0, 100))
ax.set_title("Resistencia tras llevar el plano de referencia a la resistencia")
ax.legend(loc="center right");

Al eliminar alrededor de 1 ns de línea, la oscilación entre 33 Ω y 75 Ω desaparece: de media, las dos resistencias leen a menos de un ohmio de sus valores nominales. Lo que queda es una variación lenta de unos pocos ohmios a baja frecuencia que crece hasta entre unos −8 Ω y +15 Ω para la resistencia de 75 Ω en la parte alta de la banda. Esa es la parte que un retardo puro no puede describir: una línea o un conector que no es exactamente de 50 Ω (el rizado de $|\Gamma|$ de arriba), algo de pérdidas, y la inductancia y la capacidad parásitas de una resistencia real.

Dos lecciones:

- Una calibración es válida **en un plano**. Calibra donde se va a conectar el dispositivo, con los mismos cables y adaptadores.
- Desplazar después el plano de referencia con un retardo es una herramienta habitual (los instrumentos la llaman *port extension* o *electrical delay*). Es un modelo, y solo es tan bueno como la hipótesis de «línea ideal de 50 Ω sin pérdidas» que hay detrás.

### 4.3 ¿Es buena la calibración?

Corregir las mismas tres medidas con las que calibraste no demuestra nada: el sistema tiene tres ecuaciones y tres incógnitas, así que vuelven exactamente como −1, +1 y 0 por construcción. Una prueba justa usa medidas que la calibración **no** ha visto.

Tenemos diez repeticiones de cada estándar. Calibra con las cinco primeras y corrige las cinco últimas.

In [ ]:
train = [reps[:5].mean(axis=0) for reps in (short_reps, open_reps, load_reps)]
test = [reps[5:].mean(axis=0) for reps in (short_reps, open_reps, load_reps)]
terms_train = calibration.solve_error_terms(train, ideal)

fig, ax = plt.subplots()
for name, raw, target in zip(("cortocircuito", "abierto", "carga"), test, ideal, strict=True):
    residual = np.abs(calibration.correct(raw, terms_train) - target)
    ax.semilogy(f_real_mhz, residual, label=name, linewidth=1.0)
    print(
        f"{name:<13} residuo |Γ − ideal|: mediana {np.median(residual):.4f}, "
        f"peor {residual.max():.3f} "
        f"a {f_real_mhz[residual.argmax()]:.0f} MHz"
    )
ax.set(xlabel="frecuencia (MHz)", ylabel="$|\\Gamma_{corregido} - \\Gamma_{ideal}|$ (escala log.)")
ax.set_title("Error residual en repeticiones no usadas para calibrar")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)

raw_error = np.abs(test[1] - 1)
print(
    f"\nComo comparación, el abierto en bruto se desvía {np.median(raw_error):.4f} (mediana), "
    f"{raw_error.max():.2f} (peor caso)"
)

El residuo es típicamente de unas milésimas para el cortocircuito y el abierto y de unas diezmilésimas para la carga, con puntos aislados mucho peores. Este residuo ya no es error sistemático: es la **repetibilidad** de la medida (ruido, y la conexión cambiando ligeramente entre repeticiones). La calibración no puede bajar de ahí. El cuaderno 05 estudia la repetibilidad con detalle.

Fíjate en lo que esta prueba *no* comprueba. Los mismos tres estándares físicos están a ambos lados, así que si el «abierto» no es realmente $\Gamma = +1$, esta prueba no lo notará. Eso es la sección 6.

## 5. La misma calibración con scikit-rf

No escribirás este resolvedor cada vez. `scikit-rf` implementa la calibración de un puerto como `skrf.calibration.OnePort`. Trabaja con objetos `Network` en lugar de vectores sueltos, así que primero envolvemos nuestros datos.

In [ ]:
frequency = rf.Frequency.from_f(f_real, unit="Hz")


def as_network(gamma, name):
    return rf.Network(
        frequency=frequency,
        s=np.asarray(gamma, dtype=complex) * np.ones(len(f_real)),
        z0=50,
        name=name,
    )


names = ("short", "open", "load")
cal = rf.calibration.OnePort(
    measured=[as_network(m, name) for m, name in zip(measured, names, strict=True)],
    ideals=[as_network(g, name) for g, name in zip(ideal, names, strict=True)],
)
cal.run()

# Las claves de cal.coefs son los nombres en inglés que usa scikit-rf.
for ours, key in zip(terms, ("directivity", "source match", "reflection tracking"), strict=True):
    print(
        f"{key:<20} mayor diferencia con nuestro resolvedor: "
        f"{np.abs(cal.coefs[key] - ours).max():.1e}"
    )

corrected_skrf = cal.apply_cal(as_network(duts["75 Ω"]["raw"], "75 ohm")).s[:, 0, 0]
assert np.allclose(corrected_skrf, duts["75 Ω"]["corrected"])
print("La resistencia de 75 Ω corregida es la misma con ambos")

Los mismos términos de error, el mismo resultado corregido. La biblioteca hace lo que tú has hecho a mano, lo que significa que ahora sabes qué hace. A partir de aquí, usa lo que te resulte más cómodo.

> **Una nota sobre nombres.** El repositorio de investigación del que proceden estas medidas, [nanovna-calibration](https://github.com/telmomm/nanovna-calibration), resuelve el mismo sistema pero guarda su tercera incógnita como `er`, con el modelo escrito como $\Gamma_m = (e_d + e_r\Gamma)/(1 - e_s\Gamma)$. Compáralo con la sección 1: ese `er` es $-\Delta_e = e_{10}e_{01} - e_{00}e_{11}$, que se parece al seguimiento de reflexión pero no es idéntico. El $\Gamma$ corregido es el mismo en ambos casos. El ejercicio 3 te pide comprobarlo.

## 6. ¿Y si los estándares no son ideales?

Una calibración le dice al instrumento «esto es $\Gamma = +1$». El instrumento te cree. Si el abierto que conectaste no es realmente $+1$, todas las medidas posteriores heredan ese error.

Los estándares reales no son ideales. Un abierto tiene una pequeña capacidad a masa en su extremo; un cortocircuito tiene una pequeña inductancia; una carga no es exactamente de 50 Ω. Podemos probarlo en el instrumento sintético, donde controlamos la verdad: calibra con estándares ligeramente imperfectos, una vez *fingiendo* que son ideales y otra *diciéndole al resolvedor lo que son en realidad*.

In [ ]:
omega = 2 * np.pi * f_hz
real_short = z_to_gamma(1j * omega * 0.3e-9)  # 0.3 nH en serie
real_open = z_to_gamma(1 / (1j * omega * 0.1e-12))  # 0.1 pF a masa
real_load = z_to_gamma(51.5) * ones  # 51.5 Ω en lugar de 50 Ω
real_sol = [real_short, real_open, real_load]

measured_real_sol = [apply_error_model(gamma, true_terms) for gamma in real_sol]
terms_assumed_ideal = solve_error_terms(measured_real_sol, ideal_sol)
terms_defined = solve_error_terms(measured_real_sol, real_sol)

wrong = gamma_to_z(correct(gamma_raw, terms_assumed_ideal))
right = gamma_to_z(correct(gamma_raw, terms_defined))

fig, (ax_re, ax_im) = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, part, name in (
    (ax_re, np.real, "resistencia, Re Z (Ω)"),
    (ax_im, np.imag, "reactancia, Im Z (Ω)"),
):
    ax.plot(f_mhz, part(wrong), color=plotting.SERIES[1], label="estándares supuestos ideales")
    ax.plot(f_mhz, part(right), color=plotting.SERIES[0], label="estándares descritos como son")
    ax.set(xlabel="frecuencia (MHz)", ylabel=name)
ax_re.set_ylim(70, 76)
ax_im.set_ylim(-1, 4)
ax_re.set_title("Resistencia de 75 Ω tras calibrar con estándares imperfectos")
ax_im.legend(loc="upper left")

print(
    f"Supuestos ideales: Re Z de {wrong.real.min():.1f} a {wrong.real.max():.1f} Ω, "
    f"Im Z de {wrong.imag.min():.1f} a {wrong.imag.max():.1f} Ω"
)

Con los estándares bien descritos el resultado vuelve a ser exacto. Con la hipótesis «ideal», la resistencia queda más de 2 Ω por debajo a todas las frecuencias (la carga era de 51.5 Ω y la llamamos 50 Ω), y la resistencia gana una reactancia que no tiene, que crece con la frecuencia porque 0.1 pF y 0.3 nH importan cada vez más.

Por eso un kit de calibración viene con una **definición**: números que describen cómo se comporta realmente cada estándar (retardo, capacidad de borde, inductancia). El resolvedor que has escrito ya los acepta: `actual` puede ser cualquier $\Gamma$ conocido, no solo −1, +1 y 0. Para los datos reales de la sección 4 no tenemos esa definición, así que supusimos estándares ideales, y esa hipótesis limita la exactitud de todo lo que hemos corregido.

## Comprueba lo que has entendido

1. Con estándares ideales el sistema tiene una forma cerrada sencilla. ¿Cuánto vale $e_{00}$?

<details><summary>Respuesta</summary>

Pon $\Gamma = 0$ en el modelo: $\Gamma_m = e_{00}$. La directividad es simplemente la lectura en bruto de la carga. El cortocircuito y el abierto dan entonces dos ecuaciones para los otros dos términos.
</details>

2. Calibras y después añades un cable de 10 cm entre el puerto y tu dispositivo. ¿Cuáles de tus resultados siguen siendo correctos: $|\Gamma|$, la fase de $\Gamma$, la impedancia?

<details><summary>Respuesta</summary>

Si el cable fuera una línea de 50 Ω perfecta y sin pérdidas, $|\Gamma|$ seguiría siendo correcto y la fase no, así que la impedancia sería errónea (sección 4.2). Un cable real tiene además pérdidas y su propia desadaptación, así que incluso $|\Gamma|$ se degrada. El procedimiento correcto es calibrar al final del cable.
</details>

3. ¿Por qué «corregir los estándares y comprobar que salen −1, +1 y 0» no es una prueba válida de una calibración?

<details><summary>Respuesta</summary>

Tres ecuaciones, tres incógnitas: la solución reproduce exactamente las tres medidas de calibración, sean cuales sean. Obtendrías −1, +1 y 0 aunque hubieras conectado piezas equivocadas. Una prueba necesita datos independientes: otras repeticiones (sección 4.3) o, mejor, otro dispositivo conocido.
</details>

4. En la sección 4.3 la carga tiene un residuo mucho menor que el cortocircuito y el abierto. Sugiere un motivo.

<details><summary>Respuesta</summary>

Los errores de seguimiento y de desadaptación de fuente, y cualquier cambio en ellos entre repeticiones, se multiplican por $\Gamma$. Para la carga $\Gamma \approx 0$, así que apenas se notan; para el cortocircuito y el abierto $|\Gamma| = 1$, así que se notan por completo.
</details>

## Ejercicios

Las soluciones resueltas están en [`exercises/solutions/03_solutions.ipynb`](../exercises/solutions/03_solutions.ipynb).

1. **Forma cerrada.** Para estándares ideales, deduce a mano $e_{11}$ y $e_{10}e_{01}$ a partir de las ecuaciones del cortocircuito y del abierto, impleméntalas sin `np.linalg.solve` y compáralas con `solve_error_terms` en los datos reales.
2. **Menos repeticiones.** Repite la sección 4.3 calibrando con una sola repetición en lugar de cinco. ¿Cómo cambia el residuo mediano? ¿Qué dice eso del valor de promediar?
3. **Dos parametrizaciones.** Calcula `er = tracking - directivity * source_match` a partir de los términos de error reales. Representa juntos $|e_r|$ y $|e_{10}e_{01}|$. ¿Dónde difieren más, y por qué ahí? Después comprueba que $(\Gamma_m - e_d)/(e_r + e_s\Gamma_m)$ da el mismo $\Gamma$ corregido que `correct`.
4. **Un cuarto estándar.** Añade la resistencia de 75 Ω como cuarto dispositivo conocido en el instrumento sintético. El sistema tiene ahora cuatro ecuaciones y tres incógnitas. Resuélvelo con `np.linalg.lstsq`. Añade un poco de ruido aleatorio a las cuatro «medidas» y compara el error de las soluciones con tres y con cuatro estándares.
5. **¿Cuánto importa el abierto?** En la sección 6, cambia la capacidad de borde del abierto de 0.1 pF a 0.05 pF y a 0.2 pF. A 1.45 GHz, ¿cuántos ohmios de error en la resistencia de 75 Ω causa cada valor?

In [ ]:
# Tu trabajo aquí

## Resumen

- Multiplicar el modelo de error por su denominador lo hace lineal en $e_{00}$, $e_{11}$ y $\Delta_e = e_{00}e_{11} - e_{10}e_{01}$. Tres estándares conocidos dan un sistema 3×3 por frecuencia.
- La corrección es $\Gamma = \dfrac{\Gamma_m - e_{00}}{e_{10}e_{01} + e_{11}(\Gamma_m - e_{00})}$.
- Con datos sintéticos la calibración es exacta. Con datos reales está limitada por la repetibilidad y por lo bien que conozcas tus estándares.
- Una calibración es válida en el plano donde se conectaron los estándares. Todo lo que se añade después se mide como parte del dispositivo.
- `skrf.calibration.OnePort` hace el mismo cálculo.

**Siguiente:** [05 · Validación y calidad de la medida](05_validacion_calidad.ipynb) se pregunta cuánto fiarse de estos resultados. (El cuaderno 04, sobre la adquisición de tus propias medidas con un NanoVNA, es opcional y todavía no está disponible.)